# Sabha on Colab / Kaggle

Run the cells top to bottom. The last one prints **one link**. Open it on your laptop or phone.

- Works on Colab and on Kaggle (turn **Internet on** in Kaggle's notebook settings).
- The link carries a secret token. Anyone with the link can run commands on this box, so do not share it.
- While the repo is private, add a secret named `GH_TOKEN` (Colab: key icon in the left bar. Kaggle: Add-ons > Secrets).

In [ ]:
# 1. Get the code
import os, subprocess

REPO = "https://github.com/Maverick-Ansh/sabha.git"
BASE = "/content" if os.path.isdir("/content") else "/kaggle/working"
APP = f"{BASE}/sabha"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN") or ""
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN") or ""
    except Exception:
        pass

url = REPO.replace("https://", f"https://{token}@") if token else REPO
if os.path.isdir(f"{APP}/.git"):
    r = subprocess.run(["git", "-C", APP, "pull", "--ff-only"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", url, APP], capture_output=True, text=True)
print("ok" if r.returncode == 0 else r.stderr.replace(token, "***") if token else r.stderr)

In [ ]:
# 2. Install the agent harness (DeepSeek Harness). Needs Node; Colab and Kaggle ship it.
import shutil, subprocess

if shutil.which("npm") is None:
    subprocess.run("apt-get -qq install -y nodejs npm", shell=True, check=False)
r = subprocess.run("npm i -g @deepseek-ai/dsh", shell=True, capture_output=True, text=True)
print(r.stdout[-400:] or r.stderr[-800:])
print("dsh:", shutil.which("dsh"))

In [ ]:
# 3. Start the hub in the background and print the link
import subprocess, sys, time, re

LOG = f"{BASE}/sabha_hub.log"
subprocess.run("pkill -f 'python.* -m sabha' || true", shell=True)   # stop an older hub
hub = subprocess.Popen(
    [sys.executable, "-m", "sabha", "--host", "127.0.0.1", "--port", "8787",
     "--tunnel", "cloudflared", "--data", f"{BASE}/sabha_data"],
    cwd=APP, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

link = None
for _ in range(60):                       # bounded: at most ~60 s
    time.sleep(1)
    m = re.search(r"Sabha link: (\S+)", open(LOG).read())
    if m:
        link = m.group(1)
        break
    if hub.poll() is not None:
        break
print(link or ("hub did not start. Log tail:\n" + open(LOG).read()[-2000:]))

## Next

Open the link, then **Crew > Models > Launch on this box** to start a vLLM server for any
Hugging Face model, and point an agent at it.

If you would rather start a model by hand, the cell below is an example (edit, then run).
On a Tesla T4 add `VLLM_ATTENTION_BACKEND=TRITON_ATTN` and run one server per GPU.

In [ ]:
# Optional: start a model by hand, then add http://127.0.0.1:8000/v1 in Crew > Models > Add endpoint.
# import subprocess, os
# env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
# subprocess.Popen(
#     "vllm serve Qwen/Qwen3-4B --port 8000 --served-model-name qwen "
#     "--max-model-len 24576 --gpu-memory-utilization 0.93 --enable-prefix-caching "
#     "--enable-auto-tool-choice --tool-call-parser hermes",
#     shell=True, env=env, stdout=open(f"{BASE}/vllm.log", "w"), stderr=subprocess.STDOUT,
#     start_new_session=True)

In [ ]:
# Hub log (run any time to see what the hub is doing)
print(open(LOG).read()[-3000:])